In [26]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.utils import resample
from sklearn.metrics import root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, RBF, RationalQuadratic
import time
import torch
from torch.utils.data import DataLoader, TensorDataset
from src.models.mlp import MLP, create_mlp_pytorch, EarlyStopping
from src.models.resnet import ResBlock, ResNet
import torch.optim as optim

In [27]:
descriptor_df = pd.read_csv('data/delaney/external_descriptors.csv')
smiles_df = pd.read_csv('data/delaney/smiles.csv')
learned_descriptors_df = pd.read_csv('data/delaney/learned_predictors_0.csv')
df = pd.concat([learned_descriptors_df, descriptor_df, smiles_df], axis=1)

In [28]:
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,209,210,211,212,213,214,215,y,w,ids
0,0.000000,0.026735,0.000000,0.0,0.000000,0.004854,0.000379,0.000000,0.001336,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.040026,1.0,OC1CCCCCC1
1,0.003662,0.018595,0.000786,0.0,0.000210,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.998309,1.0,CN(C)C(=O)C
2,0.001983,0.030907,0.000000,0.0,0.000151,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.264973,1.0,Clc1ccc(Cl)c(Cl)c1
3,0.000000,0.020992,0.000000,0.0,0.000000,0.003060,0.000000,0.000000,0.000065,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.067538,1.0,C1CCC=CCC1
4,0.002406,0.047554,0.000846,0.0,0.000195,0.000000,0.001088,0.000221,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.476613,1.0,CNC(=O)Oc1ccccc1OC(C)C


In [29]:
df = df.drop(columns='ids')
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,208,209,210,211,212,213,214,215,y,w
0,0.000000,0.026735,0.000000,0.0,0.000000,0.004854,0.000379,0.000000,0.001336,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.040026,1.0
1,0.003662,0.018595,0.000786,0.0,0.000210,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.998309,1.0
2,0.001983,0.030907,0.000000,0.0,0.000151,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.264973,1.0
3,0.000000,0.020992,0.000000,0.0,0.000000,0.003060,0.000000,0.000000,0.000065,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.067538,1.0
4,0.002406,0.047554,0.000846,0.0,0.000195,0.000000,0.001088,0.000221,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.476613,1.0


In [30]:
df['w'].unique()

array([1.])

In [31]:
df = df.drop(columns='w')
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,207,208,209,210,211,212,213,214,215,y
0,0.000000,0.026735,0.000000,0.0,0.000000,0.004854,0.000379,0.000000,0.001336,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.040026
1,0.003662,0.018595,0.000786,0.0,0.000210,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.998309
2,0.001983,0.030907,0.000000,0.0,0.000151,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.264973
3,0.000000,0.020992,0.000000,0.0,0.000000,0.003060,0.000000,0.000000,0.000065,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.067538
4,0.002406,0.047554,0.000846,0.0,0.000195,0.000000,0.001088,0.000221,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.476613


In [32]:
X = df.drop(columns=['y'])
y = df['y']

In [33]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

In [34]:
models_df = pd.DataFrame(columns=[
    'index', 'model_type', 'hyperparams', 'rmse'
])
models_list = []
config_id = 0
base_seed = 42

In [35]:
for i in range(4):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = KNeighborsRegressor()
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': {},
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)
 

{'index': 0, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.4051515587796057}
{'index': 1, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.407772343904169}
{'index': 2, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.4238465195510037}
{'index': 3, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.4426719779782937}


In [36]:
param_grid = {
    'alpha': [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
}

for params in ParameterGrid(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = Lasso(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 4, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.001}, 'rmse': 0.27301965483991164}


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.834e+00, tolerance: 9.375e-02
  model = cd_fast.enet_coordinate_descent(
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.048e-01, tolerance: 8.149e-02
  model = cd_fast.enet_coordinate_descent(


{'index': 5, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.002}, 'rmse': 0.26966916150457887}
{'index': 6, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.005}, 'rmse': 0.29879737667384}
{'index': 7, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.01}, 'rmse': 0.2881372132592472}
{'index': 8, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.02}, 'rmse': 0.30965339022992755}
{'index': 9, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.05}, 'rmse': 0.35525285398843265}
{'index': 10, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.1}, 'rmse': 0.40229847683261294}
{'index': 11, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.2}, 'rmse': 0.4983537047182486}


In [37]:
param_grid = {
    'alpha': [1, 2, 3, 5, 7, 10, 15, 20, 30, 50]
}

for params in ParameterGrid(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = Ridge(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 12, 'model_type': 'Ridge', 'hyperparams': {'alpha': 1}, 'rmse': 0.38044156222088155}
{'index': 13, 'model_type': 'Ridge', 'hyperparams': {'alpha': 2}, 'rmse': 0.32529609645143925}
{'index': 14, 'model_type': 'Ridge', 'hyperparams': {'alpha': 3}, 'rmse': 0.2937442584025892}
{'index': 15, 'model_type': 'Ridge', 'hyperparams': {'alpha': 5}, 'rmse': 0.31450981602160394}
{'index': 16, 'model_type': 'Ridge', 'hyperparams': {'alpha': 7}, 'rmse': 0.36644245869021924}
{'index': 17, 'model_type': 'Ridge', 'hyperparams': {'alpha': 10}, 'rmse': 0.294488641122589}
{'index': 18, 'model_type': 'Ridge', 'hyperparams': {'alpha': 15}, 'rmse': 0.2922788084309066}
{'index': 19, 'model_type': 'Ridge', 'hyperparams': {'alpha': 20}, 'rmse': 0.30053161182318305}
{'index': 20, 'model_type': 'Ridge', 'hyperparams': {'alpha': 30}, 'rmse': 0.3160915204610364}
{'index': 21, 'model_type': 'Ridge', 'hyperparams': {'alpha': 50}, 'rmse': 0.2897152923613589}


In [38]:
param_grid = [
    # Polynomial kernel (3rd degree)
    {
        'kernel': ['poly'],
        'alpha': [0.1, 1, 10],
        'degree': [3],
    },
    # RBF kernel
    {
        'kernel': ['rbf'],
        'alpha': [0.1, 1, 10],
    }
]

for idx, params in enumerate(ParameterGrid(param_grid)):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = KernelRidge(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 22, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 0.1, 'degree': 3, 'kernel': 'poly'}, 'rmse': 0.25234057150256606}
{'index': 23, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 1, 'degree': 3, 'kernel': 'poly'}, 'rmse': 0.2784992456223687}
{'index': 24, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 10, 'degree': 3, 'kernel': 'poly'}, 'rmse': 0.3122919704936978}
{'index': 25, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 0.1, 'kernel': 'rbf'}, 'rmse': 0.2975870799402795}
{'index': 26, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 1, 'kernel': 'rbf'}, 'rmse': 0.36754101129788225}
{'index': 27, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 10, 'kernel': 'rbf'}, 'rmse': 0.4899977387871365}


In [39]:
param_grid = [
    {'max_depth': [3], 'n_estimators': [10]},
    {'max_depth': [5], 'n_estimators': [10]},
    {'max_depth': [5], 'n_estimators': [20]},
    {'max_depth': [5], 'n_estimators': [50]},
    {'max_depth': [8], 'n_estimators': [50]},
    {'max_depth': [8], 'n_estimators': [100]},
    {'max_depth': [5], 'n_estimators': [100]},
    {'max_depth': [8], 'n_estimators': [100]},
    {'max_depth': [16], 'n_estimators': [200]},
    {'max_depth': [32], 'n_estimators': [1000]}
]
for idx, params in enumerate(ParameterGrid(param_grid)):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = RandomForestRegressor(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 28, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 10}, 'rmse': 0.40227695903996114}
{'index': 29, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 10}, 'rmse': 0.384579687882264}
{'index': 30, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 20}, 'rmse': 0.335293153141258}
{'index': 31, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 50}, 'rmse': 0.3329709642919662}
{'index': 32, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 8, 'n_estimators': 50}, 'rmse': 0.3322227157314187}
{'index': 33, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 8, 'n_estimators': 100}, 'rmse': 0.3130427057001605}
{'index': 34, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 100}, 'rmse': 0.326188511623829}
{'index': 35, 'model_type': 'RandomForestRegressor', 'hyperpar

In [40]:
hyperparams_list = [
    {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1},
    {'max_depth': 5, 'n_estimators': 100, 'learning_rate': 0.1},
    {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.1},
    {'max_depth': 10, 'n_estimators': 200, 'learning_rate': 0.2},
    {'max_depth': 5, 'n_estimators': 500, 'learning_rate': 0.1},
    {'max_depth': 10, 'n_estimators': 500, 'learning_rate': 0.01},
    {'max_depth': 10, 'n_estimators': 1000, 'learning_rate': 0.1},
    {'max_depth': 32, 'n_estimators': 2000, 'learning_rate': 0.1}
]

for idx, params in enumerate(hyperparams_list):

    seed = base_seed + config_id

    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)

    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)

    start_time = time.time()
    model = XGBRegressor(
        max_depth=params['max_depth'],
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        n_jobs=-1,
    )

    model.fit(X_boot_scaled, y_boot)

    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    
    rmse = root_mean_squared_error(y_val, y_val_pred)
    training_time = time.time() - start_time
    
    print(f'Training time: {training_time}')
    
    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    
    config_id += 1
    
    print(model_metadata)
    
    models_list.append(model_metadata)

Training time: 0.189896821975708
{'index': 38, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1}, 'rmse': 0.30204162087496966}
Training time: 0.36505889892578125
{'index': 39, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 100, 'learning_rate': 0.1}, 'rmse': 0.28375520899913675}
Training time: 0.3015894889831543
{'index': 40, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.1}, 'rmse': 0.2682533737741385}
Training time: 1.0780723094940186
{'index': 41, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 10, 'n_estimators': 200, 'learning_rate': 0.2}, 'rmse': 0.3441501487328449}
Training time: 1.6085922718048096
{'index': 42, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 500, 'learning_rate': 0.1}, 'rmse': 0.3102789525749514}
Training time: 11.826339960098267
{'index': 43, 'model_type': 'XGBRegressor', 'hyperpar

In [41]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, RBF, RationalQuadratic

param_grid = [
    # Matern kernel models
    {
        'kernel': Matern(),  
    },
    {
        'kernel': Matern(),  
    },
    # Quadratic (RBF) kernel models  
    {
        'kernel': RBF(),  
    },
    {
        'kernel': RBF(),
    }
]

for idx, params in enumerate(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    start_time = time.time()
    model = GaussianProcessRegressor(kernel=params['kernel'])
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)
    training_time = time.time() - start_time
    print(f'Training time: {training_time}')
    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)


Training time: 1.6078951358795166
{'index': 46, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': Matern(length_scale=1, nu=1.5)}, 'rmse': 0.29507019738796403}
Training time: 1.7710623741149902
{'index': 47, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': Matern(length_scale=1, nu=1.5)}, 'rmse': 1.0436693078788146}
Training time: 2.558197021484375
{'index': 48, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': RBF(length_scale=1)}, 'rmse': 1.0467771756140185}
Training time: 2.370460033416748
{'index': 49, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': RBF(length_scale=1)}, 'rmse': 1.0467771756140185}


In [42]:
import torch.nn as nn

In [43]:
hyperparams_list = [
    {'n_layers': 2, 'layer_size': 5, 'lr': 0.01, 'l2_reg': 0.01},
    {'n_layers': 2, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1},
    {'n_layers': 3, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1},
    {'n_layers': 3, 'layer_size': 10, 'lr': 0.0005, 'l2_reg': 0.1}
]
for idx, params in enumerate(hyperparams_list):
    
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)

    params.update({
    'input_dim': X_boot.shape[1],
    'output_dim': 1
})
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)

    X_boot_tensor = torch.FloatTensor(X_boot_scaled)
    y_boot_tensor = torch.FloatTensor(y_boot.values)
    train_dataset = TensorDataset(X_boot_tensor, y_boot_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    X_val_scaled = scaler.transform(X_val)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val.values)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32)

    # Create MLP
    model, optimizer = create_mlp_pytorch(
        params['input_dim'],
        params['output_dim'], 
        n_layers=params['n_layers'],
        layer_size=params['layer_size'],
        lr=params['lr'],
        l2_reg=params['l2_reg'])
    
    # Training code would go here
    # model.train() ... etc.
    # Basic EarlyStopping for 1000 epochs
    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()
  
    train_losses = []
    val_losses = []
    
    for epoch in range(1000):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output, target).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{1000}, '
                f'Train Loss: {avg_train_loss:.4f}, '
                f'Val Loss: {avg_val_loss:.4f}')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print("Early stopping triggered. Restoring best model...")
            # Restore the best model
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            predictions = model(data)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate RMSE
    rmse = root_mean_squared_error(all_targets, all_predictions)
    
    # Store metadata
    model_metadata = {
        'index': config_id,
        'model_type': 'MLP_PyTorch',
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    print(model_metadata)
    models_list.append(model_metadata)

Epoch 1/1000, Train Loss: 1.0592, Val Loss: 1.1333
Validation loss decreased (inf -> 1.133303). Saving model...
Epoch 2/1000, Train Loss: 0.9895, Val Loss: 1.1199
Validation loss decreased (1.133303 -> 1.119882). Saving model...
Epoch 3/1000, Train Loss: 0.9832, Val Loss: 1.1191
EarlyStopping counter: 1 out of 20
Epoch 4/1000, Train Loss: 0.9866, Val Loss: 1.1218
EarlyStopping counter: 2 out of 20
Epoch 5/1000, Train Loss: 0.9973, Val Loss: 1.1154
Validation loss decreased (1.119882 -> 1.115410). Saving model...
Epoch 6/1000, Train Loss: 1.0002, Val Loss: 1.1165
EarlyStopping counter: 1 out of 20


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect result

Epoch 7/1000, Train Loss: 1.0253, Val Loss: 1.1199
EarlyStopping counter: 2 out of 20
Epoch 8/1000, Train Loss: 1.0177, Val Loss: 1.1169
EarlyStopping counter: 3 out of 20
Epoch 9/1000, Train Loss: 1.0028, Val Loss: 1.1196
EarlyStopping counter: 4 out of 20
Epoch 10/1000, Train Loss: 1.0012, Val Loss: 1.1211
EarlyStopping counter: 5 out of 20
Epoch 11/1000, Train Loss: 0.9879, Val Loss: 1.1180
EarlyStopping counter: 6 out of 20
Epoch 12/1000, Train Loss: 1.0129, Val Loss: 1.1183
EarlyStopping counter: 7 out of 20
Epoch 13/1000, Train Loss: 0.9974, Val Loss: 1.1193
EarlyStopping counter: 8 out of 20
Epoch 14/1000, Train Loss: 0.9895, Val Loss: 1.1147
EarlyStopping counter: 9 out of 20
Epoch 15/1000, Train Loss: 0.9934, Val Loss: 1.1205
EarlyStopping counter: 10 out of 20
Epoch 16/1000, Train Loss: 0.9856, Val Loss: 1.1140
Validation loss decreased (1.115410 -> 1.113974). Saving model...
Epoch 17/1000, Train Loss: 1.0202, Val Loss: 1.1203
EarlyStopping counter: 1 out of 20
Epoch 18/1000,

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect result

Epoch 6/1000, Train Loss: 1.0664, Val Loss: 1.1120
EarlyStopping counter: 4 out of 20
Epoch 7/1000, Train Loss: 1.0788, Val Loss: 1.1126
EarlyStopping counter: 5 out of 20
Epoch 8/1000, Train Loss: 1.0844, Val Loss: 1.1135
EarlyStopping counter: 6 out of 20
Epoch 9/1000, Train Loss: 1.0787, Val Loss: 1.1154
EarlyStopping counter: 7 out of 20
Epoch 10/1000, Train Loss: 1.0708, Val Loss: 1.1114
EarlyStopping counter: 8 out of 20
Epoch 11/1000, Train Loss: 1.1157, Val Loss: 1.1123
EarlyStopping counter: 9 out of 20
Epoch 12/1000, Train Loss: 1.0820, Val Loss: 1.1131
EarlyStopping counter: 10 out of 20
Epoch 13/1000, Train Loss: 1.0695, Val Loss: 1.1141
EarlyStopping counter: 11 out of 20
Epoch 14/1000, Train Loss: 1.0986, Val Loss: 1.1123
EarlyStopping counter: 12 out of 20
Epoch 15/1000, Train Loss: 1.0640, Val Loss: 1.1120
EarlyStopping counter: 13 out of 20
Epoch 16/1000, Train Loss: 1.1275, Val Loss: 1.1128
EarlyStopping counter: 14 out of 20
Epoch 17/1000, Train Loss: 1.0846, Val Los

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect result

Epoch 3/1000, Train Loss: 1.0208, Val Loss: 1.1100
EarlyStopping counter: 2 out of 20
Epoch 4/1000, Train Loss: 1.0246, Val Loss: 1.1094
EarlyStopping counter: 3 out of 20
Epoch 5/1000, Train Loss: 1.0446, Val Loss: 1.1099
EarlyStopping counter: 4 out of 20
Epoch 6/1000, Train Loss: 1.0064, Val Loss: 1.1095
EarlyStopping counter: 5 out of 20
Epoch 7/1000, Train Loss: 1.0146, Val Loss: 1.1103
EarlyStopping counter: 6 out of 20
Epoch 8/1000, Train Loss: 1.0018, Val Loss: 1.1104
EarlyStopping counter: 7 out of 20
Epoch 9/1000, Train Loss: 1.0199, Val Loss: 1.1102
EarlyStopping counter: 8 out of 20
Epoch 10/1000, Train Loss: 1.0260, Val Loss: 1.1108
EarlyStopping counter: 9 out of 20
Epoch 11/1000, Train Loss: 1.0262, Val Loss: 1.1111
EarlyStopping counter: 10 out of 20
Epoch 12/1000, Train Loss: 1.0218, Val Loss: 1.1109
EarlyStopping counter: 11 out of 20
Epoch 13/1000, Train Loss: 1.0439, Val Loss: 1.1118
EarlyStopping counter: 12 out of 20
Epoch 14/1000, Train Loss: 1.0128, Val Loss: 1.

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect result

Epoch 4/1000, Train Loss: 0.9557, Val Loss: 1.1165
Validation loss decreased (1.118089 -> 1.116539). Saving model...
Epoch 5/1000, Train Loss: 0.9614, Val Loss: 1.1166
EarlyStopping counter: 1 out of 20
Epoch 6/1000, Train Loss: 0.9689, Val Loss: 1.1161
EarlyStopping counter: 2 out of 20
Epoch 7/1000, Train Loss: 0.9672, Val Loss: 1.1166
EarlyStopping counter: 3 out of 20
Epoch 8/1000, Train Loss: 0.9730, Val Loss: 1.1160
EarlyStopping counter: 4 out of 20
Epoch 9/1000, Train Loss: 0.9701, Val Loss: 1.1155
EarlyStopping counter: 5 out of 20
Epoch 10/1000, Train Loss: 0.9637, Val Loss: 1.1158
EarlyStopping counter: 6 out of 20
Epoch 11/1000, Train Loss: 0.9632, Val Loss: 1.1153
Validation loss decreased (1.116539 -> 1.115304). Saving model...
Epoch 12/1000, Train Loss: 0.9645, Val Loss: 1.1155
EarlyStopping counter: 1 out of 20
Epoch 13/1000, Train Loss: 0.9663, Val Loss: 1.1153
EarlyStopping counter: 2 out of 20
Epoch 14/1000, Train Loss: 0.9835, Val Loss: 1.1157
EarlyStopping counter:

In [44]:
def create_mlp_resnet(input_dim, output_dim, block_dim, hidden_dim, num_blocks, lr, l2_reg):
    model = ResNet(
        input_dim = input_dim,
        output_dim = output_dim,
        num_blocks = num_blocks,
        hidden_dim = hidden_dim,
        block_dim = block_dim
    )
    optimizer = optim.Adam(model.parameters(), lr, weight_decay=l2_reg)

    return model, optimizer

In [45]:
specified_configs = [
    # D block D hidden N blocks Learning rate L2 regularisation
    {'block_dim': 16, 'hidden_dim': 8, 'num_blocks': 2, 'lr': 0.01, 'l2_reg': 0.01},
    {'block_dim': 32, 'hidden_dim': 16, 'num_blocks': 2, 'lr': 0.01, 'l2_reg': 0.1},
    {'block_dim': 64, 'hidden_dim': 32, 'num_blocks': 3, 'lr': 0.001, 'l2_reg': 0.01},
    {'block_dim': 64, 'hidden_dim': 32, 'num_blocks': 3, 'lr': 0.001, 'l2_reg': 0.1},
]

for idx, params in enumerate(specified_configs):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    params.update({
    'input_dim': X_boot.shape[1],
    'output_dim': 1
})
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    X_boot_tensor = torch.FloatTensor(X_boot_scaled)
    y_boot_tensor = torch.FloatTensor(y_boot.values)
    train_dataset = TensorDataset(X_boot_tensor, y_boot_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    X_val_scaled = scaler.transform(X_val)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val.values)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32)

    model, optimizer = create_mlp_resnet(
        input_dim = params['input_dim'], 
        output_dim = params['output_dim'],
        num_blocks=params['num_blocks'],
        hidden_dim=params['hidden_dim'],
        block_dim=params['block_dim'],
        lr=params['lr'],
        l2_reg=params['l2_reg'])

    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()

    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()
  
    train_losses = []
    val_losses = []
    
    for epoch in range(1000):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output, target).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{1000}, '
                f'Train Loss: {avg_train_loss:.4f}, '
                f'Val Loss: {avg_val_loss:.4f}')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print("Early stopping triggered. Restoring best model...")
            # Restore the best model
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            predictions = model(data)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate RMSE
    rmse = root_mean_squared_error(all_targets, all_predictions)
    
    # Store metadata
    model_metadata = {
        'index': config_id,
        'model_type': 'ResNet',
        'hyperparams': None,
        'rmse': rmse
    }
    config_id += 1
    print(model_metadata)
    models_list.append(model_metadata)

Epoch 1/1000, Train Loss: 1.1137, Val Loss: 1.1530
Validation loss decreased (inf -> 1.152955). Saving model...


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect result

Epoch 2/1000, Train Loss: 1.0456, Val Loss: 1.1381
Validation loss decreased (1.152955 -> 1.138098). Saving model...
Epoch 3/1000, Train Loss: 1.0557, Val Loss: 1.1139
Validation loss decreased (1.138098 -> 1.113906). Saving model...
Epoch 4/1000, Train Loss: 1.0405, Val Loss: 1.1176
EarlyStopping counter: 1 out of 20
Epoch 5/1000, Train Loss: 1.0376, Val Loss: 1.1253
EarlyStopping counter: 2 out of 20
Epoch 6/1000, Train Loss: 1.0624, Val Loss: 1.1213
EarlyStopping counter: 3 out of 20
Epoch 7/1000, Train Loss: 1.0471, Val Loss: 1.1500
EarlyStopping counter: 4 out of 20
Epoch 8/1000, Train Loss: 1.0719, Val Loss: 1.1477
EarlyStopping counter: 5 out of 20
Epoch 9/1000, Train Loss: 1.0623, Val Loss: 1.1274
EarlyStopping counter: 6 out of 20
Epoch 10/1000, Train Loss: 1.0463, Val Loss: 1.1249
EarlyStopping counter: 7 out of 20
Epoch 11/1000, Train Loss: 1.0312, Val Loss: 1.1246
EarlyStopping counter: 8 out of 20
Epoch 12/1000, Train Loss: 1.0338, Val Loss: 1.1204
EarlyStopping counter: 9

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect result

Epoch 3/1000, Train Loss: 1.0495, Val Loss: 1.1126
Validation loss decreased (1.134975 -> 1.112643). Saving model...
Epoch 4/1000, Train Loss: 1.0887, Val Loss: 1.1113
Validation loss decreased (1.112643 -> 1.111304). Saving model...
Epoch 5/1000, Train Loss: 1.0308, Val Loss: 1.1108
EarlyStopping counter: 1 out of 20
Epoch 6/1000, Train Loss: 1.0343, Val Loss: 1.1111
EarlyStopping counter: 2 out of 20
Epoch 7/1000, Train Loss: 1.0331, Val Loss: 1.1101
Validation loss decreased (1.111304 -> 1.110066). Saving model...
Epoch 8/1000, Train Loss: 1.0605, Val Loss: 1.1085
Validation loss decreased (1.110066 -> 1.108468). Saving model...
Epoch 9/1000, Train Loss: 1.0360, Val Loss: 1.1104
EarlyStopping counter: 1 out of 20
Epoch 10/1000, Train Loss: 1.0396, Val Loss: 1.1099
EarlyStopping counter: 2 out of 20
Epoch 11/1000, Train Loss: 1.0149, Val Loss: 1.1108
EarlyStopping counter: 3 out of 20
Epoch 12/1000, Train Loss: 1.0971, Val Loss: 1.1106
EarlyStopping counter: 4 out of 20
Epoch 13/1000

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect result

Epoch 4/1000, Train Loss: 0.9269, Val Loss: 1.1113
Validation loss decreased (1.119075 -> 1.111343). Saving model...
Epoch 5/1000, Train Loss: 0.9320, Val Loss: 1.1143
EarlyStopping counter: 1 out of 20
Epoch 6/1000, Train Loss: 0.9576, Val Loss: 1.1249
EarlyStopping counter: 2 out of 20
Epoch 7/1000, Train Loss: 0.9284, Val Loss: 1.1298
EarlyStopping counter: 3 out of 20
Epoch 8/1000, Train Loss: 0.9795, Val Loss: 1.1105
EarlyStopping counter: 4 out of 20
Epoch 9/1000, Train Loss: 0.9422, Val Loss: 1.1193
EarlyStopping counter: 5 out of 20
Epoch 10/1000, Train Loss: 0.9364, Val Loss: 1.1161
EarlyStopping counter: 6 out of 20
Epoch 11/1000, Train Loss: 1.0031, Val Loss: 1.1246
EarlyStopping counter: 7 out of 20
Epoch 12/1000, Train Loss: 0.9402, Val Loss: 1.1194
EarlyStopping counter: 8 out of 20
Epoch 13/1000, Train Loss: 0.9358, Val Loss: 1.1101
Validation loss decreased (1.111343 -> 1.110127). Saving model...
Epoch 14/1000, Train Loss: 0.9238, Val Loss: 1.1138
EarlyStopping counter:

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect result

Epoch 1/1000, Train Loss: 0.9295, Val Loss: 1.1425
Validation loss decreased (inf -> 1.142530). Saving model...
Epoch 2/1000, Train Loss: 0.9505, Val Loss: 1.1211
Validation loss decreased (1.142530 -> 1.121106). Saving model...
Epoch 3/1000, Train Loss: 0.9301, Val Loss: 1.1421
EarlyStopping counter: 1 out of 20
Epoch 4/1000, Train Loss: 0.9564, Val Loss: 1.1312
EarlyStopping counter: 2 out of 20
Epoch 5/1000, Train Loss: 0.9701, Val Loss: 1.1370
EarlyStopping counter: 3 out of 20
Epoch 6/1000, Train Loss: 0.9741, Val Loss: 1.1212
EarlyStopping counter: 4 out of 20
Epoch 7/1000, Train Loss: 0.9245, Val Loss: 1.1279
EarlyStopping counter: 5 out of 20
Epoch 8/1000, Train Loss: 0.9150, Val Loss: 1.1212
EarlyStopping counter: 6 out of 20
Epoch 9/1000, Train Loss: 0.9158, Val Loss: 1.1277
EarlyStopping counter: 7 out of 20
Epoch 10/1000, Train Loss: 0.9066, Val Loss: 1.1282
EarlyStopping counter: 8 out of 20
Epoch 11/1000, Train Loss: 0.9347, Val Loss: 1.1224
EarlyStopping counter: 9 out o

In [46]:
models_list

[{'index': 0,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.4051515587796057},
 {'index': 1,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.407772343904169},
 {'index': 2,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.4238465195510037},
 {'index': 3,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.4426719779782937},
 {'index': 4,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.001},
  'rmse': 0.27301965483991164},
 {'index': 5,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.002},
  'rmse': 0.26966916150457887},
 {'index': 6,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.005},
  'rmse': 0.29879737667384},
 {'index': 7,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.01},
  'rmse': 0.2881372132592472},
 {'index': 8,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.02},
  'rmse': 0.30965339022992755},
 {'index': 9,
  'model_type': 'Lasso',
  'hyperparams': {

In [47]:
models_df = pd.DataFrame(models_list)
models_df = models_df.sort_values('rmse', ascending=True).reset_index(drop=True)
final_models = models_df.head(10)
final_models

,index,model_type,hyperparams,rmse
0,22,KernelRidge,"{'alpha': 0.1, 'degree': 3, 'kernel': 'poly'}",0.252341
1,40,XGBRegressor,"{'max_depth': 3, 'n_estimators': 200, 'learnin...",0.268253
2,5,Lasso,{'alpha': 0.002},0.269669
3,4,Lasso,{'alpha': 0.001},0.273020
4,23,KernelRidge,"{'alpha': 1, 'degree': 3, 'kernel': 'poly'}",0.278499
5,39,XGBRegressor,"{'max_depth': 5, 'n_estimators': 100, 'learnin...",0.283755
6,7,Lasso,{'alpha': 0.01},0.288137
7,21,Ridge,{'alpha': 50},0.289715
8,18,Ridge,{'alpha': 15},0.292279
9,14,Ridge,{'alpha': 3},0.293744


In [48]:
def softmax(series):
    exp_x = np.exp(series - series.max())
    return exp_x / exp_x.sum()

In [49]:
final_models['weights'] = softmax(-final_models['rmse'])
final_models

/tmp/ipykernel_385388/2114309755.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_models['weights'] = softmax(-final_models['rmse'])


,index,model_type,hyperparams,rmse,weights
0,22,KernelRidge,"{'alpha': 0.1, 'degree': 3, 'kernel': 'poly'}",0.252341,0.102688
1,40,XGBRegressor,"{'max_depth': 3, 'n_estimators': 200, 'learnin...",0.268253,0.101067
2,5,Lasso,{'alpha': 0.002},0.269669,0.100924
3,4,Lasso,{'alpha': 0.001},0.273020,0.100586
4,23,KernelRidge,"{'alpha': 1, 'degree': 3, 'kernel': 'poly'}",0.278499,0.100036
5,39,XGBRegressor,"{'max_depth': 5, 'n_estimators': 100, 'learnin...",0.283755,0.099512
6,7,Lasso,{'alpha': 0.01},0.288137,0.099077
7,21,Ridge,{'alpha': 50},0.289715,0.098921
8,18,Ridge,{'alpha': 15},0.292279,0.098667
9,14,Ridge,{'alpha': 3},0.293744,0.098523


In [50]:
rmse = sum(final_models['rmse']*final_models['weights'])
rmse

0.2787842262873828